In [1]:

import numpy as np
import warnings

import pandas as pd
from pandas.core.common import SettingWithCopyWarning

warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)

In [2]:
data = pd.read_csv("data/inline_pins.csv")
data.head()

,Connector Name,Connector PartNumber,Pin Name,Pin PreferredSignal,Signal Name,Wire WireColor,Wire WireCSA,MulticoreInnerToOutter1,MulticoreInnerToOutter2,MulticoreInnerToOutter4,MulticoreInnerToOutter3,MulticoreInnerToOutter5,MulticoreInnerToOutter6,MulticoreInnerToOutter7,MulticoreInnerToOutter8,MulticoreInnerToOutter9
0,TO_141_LH_LHD_F/111_LH_LHD,NaN,19,NaN,EQ_METER_LL,V,0.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,TO_141_LH_LHD_F/111_LH_LHD,NaN,39,NaN,SHIFT_ECU_1604,G,0.35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,TO_141_LH_LHD_F/111_LH_LHD,NaN,35,NaN,EQ_F:SBW_IGCT-HV_L,B,0.35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,TO_141_LH_LHD_F/111_LH_LHD,NaN,16,NaN,EQ_FUEL_LID_OPNR_SW_IG,L,0.35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,TO_141_LH_LHD_F/111_LH_LHD,NaN,48,NaN,H/P_ECU_1431,L,0.35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
data["Signal Name"].fillna("na", inplace=True)
data["Pin PreferredSignal"].fillna("na", inplace=True)
data['Connector PartNumber'].fillna("NOPARTNUMBER", inplace=True)
data['Wire WireColor'].fillna("na", inplace=True)
data['Wire WireCSA'].fillna(0, inplace=True)
data['MulticoreInnerToOutter1'].fillna("na", inplace=True)

In [4]:
def signal_equal(row):
    if (row["Pin PreferredSignal"]== "na" and row["Signal Name"]== "na") :
        return 2
    elif row["Pin PreferredSignal"]==row["Signal Name"]:
        return 1
    else:
        return 0

In [5]:
data['check_signal']=data.apply(lambda row:signal_equal(row),axis=1)

In [6]:
connector_view=data.groupby(["Connector Name",'Pin Name']).agg({ "Pin PreferredSignal":['nunique','count'] })
connector_view.head(10)

Pin PreferredSignal      
                                     nunique count
Connector Name  Pin Name                          
TO_111_LH_F/113 1                          1     1
                2                          1     1
                3                          1     1
                4                          1     1
                5                          1     1
                6                          1     1
                7                          1     1
                8                          1     1
                9                          1     1
                10                         1     1

In [7]:
levels=connector_view.index.nlevels
for i in reversed(range(0,levels)):
    connector_view.reset_index(level=i, inplace=True)

In [8]:
connector_view.columns=['Connector Name','Pin Name','Pin Name nunique',"Pin Name count"]
connector_view.tail(10)

,Connector Name,Pin Name,Pin Name nunique,Pin Name count
1708,TO_219_F/121_TNGA_13P,4,1,1
1709,TO_219_F/121_TNGA_13P,5,1,1
1710,TO_219_F/121_TNGA_13P,6,1,1
1711,TO_219_F/121_TNGA_13P,7,1,1
1712,TO_219_F/121_TNGA_13P,8,1,1
1713,TO_219_F/121_TNGA_13P,9,1,1
1714,TO_219_F/121_TNGA_13P,10,1,1
1715,TO_219_F/121_TNGA_13P,11,1,1
1716,TO_219_F/121_TNGA_13P,12,1,1
1717,TO_219_F/121_TNGA_13P,13,1,1


In [9]:
groupped= pd.merge(connector_view,data, on=['Connector Name','Pin Name'], how='inner')
len(connector_view),len(data),len(groupped)

(1718, 1718, 1718)

In [10]:
connector_view2=data.groupby(["Connector Name"]).agg({ 'Pin Name':['nunique','count'] })
connector_view2.head()

Pin Name      
                            nunique count
Connector Name                           
TO_111_LH_F/113                  17    17
TO_111_LH_F/121                  28    28
TO_111_LH_F/143_3P                3     3
TO_111_LH_F/161_LH               53    53
TO_111_LH_LHD_F/141_LH_LHD       49    49

In [11]:
levels=connector_view2.index.nlevels
for i in reversed(range(0,levels)):
    connector_view2.reset_index(level=i, inplace=True)

In [12]:
connector_view2.columns=['Connector Name','Num_Pin_unique','Num_Pins']

In [13]:
connector_view2['more_signal_in_pin']=connector_view2["Num_Pin_unique"]!=connector_view2["Num_Pins"]
connector_view2.head(25)

,Connector Name,Num_Pin_unique,Num_Pins,more_signal_in_pin
0,TO_111_LH_F/113,17,17,False
1,TO_111_LH_F/121,28,28,False
2,TO_111_LH_F/143_3P,3,3,False
3,TO_111_LH_F/161_LH,53,53,False
4,TO_111_LH_LHD_F/141_LH_LHD,49,49,False
5,TO_111_LH_RHD_F/141_LH_RHD,50,50,False
6,TO_111_RH_F/161_RH,41,41,False
7,TO_111_RH_LHD_F/141_RH_LHD,35,35,False
8,TO_111_RH_RHD_F/141_RH_RHD,42,42,False
9,TO_113_F/111_LH,17,17,False


In [14]:
groupped2= pd.merge(groupped,connector_view2, on=['Connector Name'], how='inner')
len(groupped2),len(groupped)

(1718, 1718)

In [15]:
groupped2.head(15)

,Connector Name,Pin Name,Pin Name nunique,Pin Name count,Connector PartNumber,Pin PreferredSignal,Signal Name,Wire WireColor,Wire WireCSA,MulticoreInnerToOutter1,...,MulticoreInnerToOutter3,MulticoreInnerToOutter5,MulticoreInnerToOutter6,MulticoreInnerToOutter7,MulticoreInnerToOutter8,MulticoreInnerToOutter9,check_signal,Num_Pin_unique,Num_Pins,more_signal_in_pin
0,TO_111_LH_F/113,1,1,1,NOPARTNUMBER,na,EMB_PVM_ECU_064,W,1.00,MC137112,...,NaN,NaN,NaN,NaN,NaN,NaN,0,17,17,False
1,TO_111_LH_F/113,2,1,1,NOPARTNUMBER,na,EMB_PVM_ECU_065,W,1.00,MC137112,...,NaN,NaN,NaN,NaN,NaN,NaN,0,17,17,False
2,TO_111_LH_F/113,3,1,1,NOPARTNUMBER,na,EMB_PVM_ECU_066,W,1.00,MC137112,...,NaN,NaN,NaN,NaN,NaN,NaN,0,17,17,False
3,TO_111_LH_F/113,4,1,1,NOPARTNUMBER,na,EMB_PVM_ECU_067,W,1.00,MC137112,...,NaN,NaN,NaN,NaN,NaN,NaN,0,17,17,False
4,TO_111_LH_F/113,5,1,1,NOPARTNUMBER,na,EMB_FRONT_CAMERA-PVM_063,W,1.00,MC137112,...,NaN,NaN,NaN,NaN,NaN,NaN,0,17,17,False
5,TO_111_LH_F/113,6,1,1,NOPARTNUMBER,na,TW_CLEARANCE_SNR_ECU_075,LA-R,0.50,MC137529,...,NaN,NaN,NaN,NaN,NaN,NaN,0,17,17,False
6,TO_111_LH_F/113,7,1,1,NOPARTNUMBER,na,TW_CLEARANCE_SNR_ECU_076,LA-G,0.50,MC137529,...,NaN,NaN,NaN,NaN,NaN,NaN,0,17,17,False
7,TO_111_LH_F/113,8,1,1,NOPARTNUMBER,na,TW_CLEARANCE_SNR_ECU_077,LA-G,0.50,MC137533,...,NaN,NaN,NaN,NaN,NaN,NaN,0,17,17,False
8,TO_111_LH_F/113,9,1,1,NOPARTNUMBER,na,TW_CLEARANCE_SNR_ECU_078,LA-R,0.50,MC137533,...,NaN,NaN,NaN,NaN,NaN,NaN,0,17,17,False
9,TO_111_LH_F/113,10,1,1,NOPARTNUMBER,na,TW_CERTIFICATION_ECU_278,G,0.35,MC137622,...,NaN,NaN,NaN,NaN,NaN,NaN,0,17,17,False


In [16]:
def category_signal(row):
    if row["Signal Name"][:5]=='BATT_':
        return 1
    elif row["Signal Name"][:4]=='CAN_':
        return 2
    elif row["Signal Name"][:6]=='CANFD_':
        return 3
    elif row["Signal Name"][:4]=='GND_':
        return 4
    elif row["Signal Name"][:9]=='GNDR_GND_':
        return 5
    elif row["Signal Name"][:3]=='HV_':
        return 6
    else:
        return 7

In [17]:
groupped2["signal_cat"]= groupped2.apply(lambda row:category_signal(row), axis=1 )

In [18]:
groupped2["signal_cat"].unique()

array([7, 4, 2, 1, 3, 6], dtype=int64)

In [19]:
def category_signal2(row):
    if row["Signal Name"][-3:]=='_HI' or row["Signal Name"][-3:]=='_LO':
        return 1
    else:
        return 0

In [20]:
groupped2["signal_multicore"]= groupped2.apply(lambda row:category_signal2(row), axis=1 )

In [21]:
groupped2[["Signal Name","signal_cat","signal_multicore"]].tail(25)

,Signal Name,signal_cat,signal_multicore
1693,EQ_F:PHV_BATT_L,7,0
1694,GND_CHG_INDICATOR_GND,4,0
1695,GND_HV_BAT_JB_GND,4,0
1696,GND_PHV_BAT_SYS_GND3,4,0
1697,GND_PHV_BAT_SYS_HGND0,4,0
1698,GND_PHV_BAT_SYS_HGND1,4,0
1699,HV_ECU-PHV_0059,6,0
1700,HV_ECU-PHV_2347,6,0
1701,HV_ECU-PHV_2355,6,0
1702,HV_ECU-PHV_2357,6,0


In [22]:
groupped2.columns

Index(['Connector Name', 'Pin Name', 'Pin Name nunique', 'Pin Name count',
       'Connector PartNumber', 'Pin PreferredSignal', 'Signal Name',
       'Wire WireColor', 'Wire WireCSA', 'MulticoreInnerToOutter1',
       'MulticoreInnerToOutter2', 'MulticoreInnerToOutter4',
       'MulticoreInnerToOutter3', 'MulticoreInnerToOutter5',
       'MulticoreInnerToOutter6', 'MulticoreInnerToOutter7',
       'MulticoreInnerToOutter8', 'MulticoreInnerToOutter9', 'check_signal',
       'Num_Pin_unique', 'Num_Pins', 'more_signal_in_pin', 'signal_cat',
       'signal_multicore'],
      dtype='object')

In [23]:
def myfunc_data(data):
    # function groupping by 
    data['Cat_Signal Name'] = data["Signal Name"].astype('category').cat.codes
    
    return data

In [24]:
def myfunc_data(data):
    # function groupping by 
    data['Cat_Signal Name'] = data["Signal Name"].astype('category').cat.codes
    
    return data
groupped2=groupped2.groupby('Num_Pins').apply(myfunc_data)
groupped2.head(18)

,Connector Name,Pin Name,Pin Name nunique,Pin Name count,Connector PartNumber,Pin PreferredSignal,Signal Name,Wire WireColor,Wire WireCSA,MulticoreInnerToOutter1,...,MulticoreInnerToOutter7,MulticoreInnerToOutter8,MulticoreInnerToOutter9,check_signal,Num_Pin_unique,Num_Pins,more_signal_in_pin,signal_cat,signal_multicore,Cat_Signal Name
0,TO_111_LH_F/113,1,1,1,NOPARTNUMBER,na,EMB_PVM_ECU_064,W,1.00,MC137112,...,NaN,NaN,NaN,0,17,17,False,7,0,3
1,TO_111_LH_F/113,2,1,1,NOPARTNUMBER,na,EMB_PVM_ECU_065,W,1.00,MC137112,...,NaN,NaN,NaN,0,17,17,False,7,0,4
2,TO_111_LH_F/113,3,1,1,NOPARTNUMBER,na,EMB_PVM_ECU_066,W,1.00,MC137112,...,NaN,NaN,NaN,0,17,17,False,7,0,5
3,TO_111_LH_F/113,4,1,1,NOPARTNUMBER,na,EMB_PVM_ECU_067,W,1.00,MC137112,...,NaN,NaN,NaN,0,17,17,False,7,0,6
4,TO_111_LH_F/113,5,1,1,NOPARTNUMBER,na,EMB_FRONT_CAMERA-PVM_063,W,1.00,MC137112,...,NaN,NaN,NaN,0,17,17,False,7,0,2
5,TO_111_LH_F/113,6,1,1,NOPARTNUMBER,na,TW_CLEARANCE_SNR_ECU_075,LA-R,0.50,MC137529,...,NaN,NaN,NaN,0,17,17,False,7,0,13
6,TO_111_LH_F/113,7,1,1,NOPARTNUMBER,na,TW_CLEARANCE_SNR_ECU_076,LA-G,0.50,MC137529,...,NaN,NaN,NaN,0,17,17,False,7,0,14
7,TO_111_LH_F/113,8,1,1,NOPARTNUMBER,na,TW_CLEARANCE_SNR_ECU_077,LA-G,0.50,MC137533,...,NaN,NaN,NaN,0,17,17,False,7,0,15
8,TO_111_LH_F/113,9,1,1,NOPARTNUMBER,na,TW_CLEARANCE_SNR_ECU_078,LA-R,0.50,MC137533,...,NaN,NaN,NaN,0,17,17,False,7,0,16
9,TO_111_LH_F/113,10,1,1,NOPARTNUMBER,na,TW_CERTIFICATION_ECU_278,G,0.35,MC137622,...,NaN,NaN,NaN,0,17,17,False,7,0,11


In [25]:
def get_dictionary(data):
    c = data["Signal Name"].astype('category')
    data['dicc_signals'] = str(dict(enumerate(c.cat.categories)))
    return data

In [26]:
groupped2=groupped2.groupby("Num_Pins").apply(get_dictionary)
groupped2.head(18)

,Connector Name,Pin Name,Pin Name nunique,Pin Name count,Connector PartNumber,Pin PreferredSignal,Signal Name,Wire WireColor,Wire WireCSA,MulticoreInnerToOutter1,...,MulticoreInnerToOutter8,MulticoreInnerToOutter9,check_signal,Num_Pin_unique,Num_Pins,more_signal_in_pin,signal_cat,signal_multicore,Cat_Signal Name,dicc_signals
0,TO_111_LH_F/113,1,1,1,NOPARTNUMBER,na,EMB_PVM_ECU_064,W,1.00,MC137112,...,NaN,NaN,0,17,17,False,7,0,3,"{0: 'CLEARANCE_SNR_ECU_1444', 1: 'CLEARANCE_SN..."
1,TO_111_LH_F/113,2,1,1,NOPARTNUMBER,na,EMB_PVM_ECU_065,W,1.00,MC137112,...,NaN,NaN,0,17,17,False,7,0,4,"{0: 'CLEARANCE_SNR_ECU_1444', 1: 'CLEARANCE_SN..."
2,TO_111_LH_F/113,3,1,1,NOPARTNUMBER,na,EMB_PVM_ECU_066,W,1.00,MC137112,...,NaN,NaN,0,17,17,False,7,0,5,"{0: 'CLEARANCE_SNR_ECU_1444', 1: 'CLEARANCE_SN..."
3,TO_111_LH_F/113,4,1,1,NOPARTNUMBER,na,EMB_PVM_ECU_067,W,1.00,MC137112,...,NaN,NaN,0,17,17,False,7,0,6,"{0: 'CLEARANCE_SNR_ECU_1444', 1: 'CLEARANCE_SN..."
4,TO_111_LH_F/113,5,1,1,NOPARTNUMBER,na,EMB_FRONT_CAMERA-PVM_063,W,1.00,MC137112,...,NaN,NaN,0,17,17,False,7,0,2,"{0: 'CLEARANCE_SNR_ECU_1444', 1: 'CLEARANCE_SN..."
5,TO_111_LH_F/113,6,1,1,NOPARTNUMBER,na,TW_CLEARANCE_SNR_ECU_075,LA-R,0.50,MC137529,...,NaN,NaN,0,17,17,False,7,0,13,"{0: 'CLEARANCE_SNR_ECU_1444', 1: 'CLEARANCE_SN..."
6,TO_111_LH_F/113,7,1,1,NOPARTNUMBER,na,TW_CLEARANCE_SNR_ECU_076,LA-G,0.50,MC137529,...,NaN,NaN,0,17,17,False,7,0,14,"{0: 'CLEARANCE_SNR_ECU_1444', 1: 'CLEARANCE_SN..."
7,TO_111_LH_F/113,8,1,1,NOPARTNUMBER,na,TW_CLEARANCE_SNR_ECU_077,LA-G,0.50,MC137533,...,NaN,NaN,0,17,17,False,7,0,15,"{0: 'CLEARANCE_SNR_ECU_1444', 1: 'CLEARANCE_SN..."
8,TO_111_LH_F/113,9,1,1,NOPARTNUMBER,na,TW_CLEARANCE_SNR_ECU_078,LA-R,0.50,MC137533,...,NaN,NaN,0,17,17,False,7,0,16,"{0: 'CLEARANCE_SNR_ECU_1444', 1: 'CLEARANCE_SN..."
9,TO_111_LH_F/113,10,1,1,NOPARTNUMBER,na,TW_CERTIFICATION_ECU_278,G,0.35,MC137622,...,NaN,NaN,0,17,17,False,7,0,11,"{0: 'CLEARANCE_SNR_ECU_1444', 1: 'CLEARANCE_SN..."


In [27]:
# groupped2['Cat_Signal Name'] = groupped2["Signal Name"].astype('category').cat.codes
# c = groupped2["Signal Name"].astype('category')
# d = dict(enumerate(c.cat.categories))

In [28]:
def get_key(x):
    d=eval(x['dicc_signals'])
    key_list=list(d.keys())
    val_list=list(d.values())
    try:
        ind=val_list.index(x["Pin PreferredSignal"])
        return key_list[ind]
    except:
        return len(val_list)+1

In [29]:
groupped2['Cat_PreferredSignal']= groupped2.apply(lambda row: get_key(row),axis=1)

In [30]:
c = groupped2["Signal Name"].astype('category')
d = dict(enumerate(c.cat.categories))
len(d.keys())

450

In [31]:
max_categories = len(d.keys())

In [32]:
groupped2["Cat_Wire_WireColor"]=groupped2["Wire WireColor"].astype('category').cat.codes
groupped2["Cat_Wire_WireColor_max"]=groupped2["Cat_Wire_WireColor"].max()
groupped2["Wire_WireCSA_min"]=groupped2["Wire WireCSA"].min()
groupped2["Wire_WireCSA_max"]=groupped2["Wire WireCSA"].max()
groupped2['Num_Pin_unique_max']=groupped2['Num_Pin_unique'].max()

In [34]:
def no_signal_in_pin(row):
    if row["Signal Name"]=="na":
        return 1
    else:
        return 0
    

In [35]:
groupped2["no_signal_pin"]=groupped2.apply(lambda row: no_signal_in_pin(row), axis=1)

In [38]:
groupped2.to_csv("data/final_groupped.csv", index=False)

In [41]:
def multicore_data(data):
    
    data['multicore_same'] = data["MulticoreInnerToOutter1"].astype('category').cat.codes
    
    return data

In [42]:
groupped2=groupped2.groupby('Connector Name').apply(multicore_data)

In [49]:
def is_multicore(row):
    if row['MulticoreInnerToOutter1']!="na":
        return 1
    else:
        return 0

In [50]:
groupped2['is_multicore']=groupped2.apply(lambda row: is_multicore(row), axis=1)

In [53]:
groupped2['max_categories_multicore']=groupped2['multicore_same'].max()

In [54]:
groupped2[["MulticoreInnerToOutter1",'is_multicore','multicore_same','max_categories_multicore']].head(13)

,MulticoreInnerToOutter1,is_multicore,multicore_same,max_categories_multicore
0,MC137112,1,0,8
1,MC137112,1,0,8
2,MC137112,1,0,8
3,MC137112,1,0,8
4,MC137112,1,0,8
5,MC137529,1,1,8
6,MC137529,1,1,8
7,MC137533,1,2,8
8,MC137533,1,2,8
9,MC137622,1,3,8


In [52]:
groupped2[['multicore_same']].max()

multicore_same    8
dtype: int8

In [46]:
symbols= pd.read_pickle('data/symbols_distances.pkl')
symbols.head()

,Symbol Name,Pin Name,Pin CenterY,Pin CenterX,Pin Width,Pin Height,distance_to_center,neighbors,internal_pin
0,90980-12A87,8,-5759.0,10368.0,0.0,0.0,11860.080312,"[7, 9, 20, 21, 19]",False
1,90980-12A87,10,-13439.0,10368.0,0.0,0.0,16973.571958,"[11, 9, 22, 23, 21]",True
2,90980-12A87,23,-17279.0,5760.0,0.0,0.0,18213.770642,"[22, 24, 11, 10, 12]",False
3,90980-12A87,6,1920.0,10368.0,0.0,0.0,10544.279207,"[7, 5, 18, 19, 17]",False
4,90980-12A87,15,13440.0,5760.0,0.0,0.0,14622.284363,"[14, 16, 3, 4, 2]",False


In [47]:
# symbols["Pin Name"].unique()

In [48]:
groupped2['Pin Name']=groupped2['Pin Name'].astype(str)

In [49]:
groupped2_copy=groupped2.copy()

In [50]:
symbols.columns

Index(['Symbol Name', 'Pin Name', 'Pin CenterY', 'Pin CenterX', 'Pin Width',
       'Pin Height', 'distance_to_center', 'neighbors', 'internal_pin'],
      dtype='object')

In [51]:
groupped2b=pd.merge(groupped2, symbols, left_on=['Connector PartNumber','Pin Name'],
                    right_on=['Symbol Name', 'Pin Name'], how="left")

In [52]:
groupped2b.to_csv("data/final_groupped.csv", index=False)

In [38]:
groupped2=pd.merge(groupped2, symbols, left_on=['Connector PartNumber','Pin Name'],
                    right_on=['Symbol Name', 'Pin Name'], how="left")

In [39]:
connectors={}
conn_to_save=""
for i, row in groupped2.iterrows():
    connector=groupped2.loc[i,"Connector Name"]
    pin=groupped2.loc[i,'Pin Name']
    
    if conn_to_save=="" or groupped2.loc[i-1,"Connector Name"]==connector:
        try:
        
            connectors[connector][pin]={}
        except:
            connectors[connector]={}
            connectors[connector][pin]={}
            
        
        conn_to_save="NO"
    else:
        conn_to_save=""
        try:
        
            connectors[connector][pin]={}
        except:
            connectors[connector]={}
            connectors[connector][pin]={}
    
    #Part Number at level of PIN
    PartNumber =groupped2.loc[i,'Connector PartNumber']
    connectors[connector]["PartNumber"]=PartNumber
    #prefered Signal
    PreferredSignal=groupped2.loc[i,'Pin PreferredSignal']
    connectors[connector][pin]['PreferredSignal']=PreferredSignal
    Cat_PreferredSignal=groupped2.loc[i,'Cat_PreferredSignal']
    connectors[connector][pin]['Cat_PreferredSignal']=Cat_PreferredSignal
    
    # Signal in that pin
    Signal =groupped2.loc[i,'Signal Name']
    connectors[connector][pin]['Signal']=Signal
    Cat_Signal =groupped2.loc[i,'Cat_Signal Name']
    connectors[connector][pin]['Cat_Signal']=Cat_Signal
    # dictionary of categories
    dicc_signals =groupped2.loc[i,'dicc_signals']
    connectors[connector][pin]['dicc_signals']=dicc_signals
    
    #Check Signal 1 preferred = Signal, 0 1 preferred != Signal, 2 both NaN 
    check_signal=groupped2.loc[i,'check_signal']
    connectors[connector][pin]['check_signal']=check_signal
    #Number of pins connector
    Num_Pins =groupped2.loc[i,'Num_Pins']
    connectors[connector][pin]['Num_Pins']=Num_Pins
    #Number of unique Pins in this connector
    Num_Pin_unique=groupped2.loc[i,'Num_Pin_unique']
    connectors[connector][pin]['Num_Pin_unique']=Num_Pin_unique
    #True if number of Pins is not the same of number of unique Pins
    more_signal_in_pin =groupped2.loc[i,'more_signal_in_pin']
    connectors[connector][pin]['more_signal_in_pin']=more_signal_in_pin
    
    #WireColor
    WireColor =groupped2.loc[i,'Wire WireColor']
    connectors[connector][pin]['WireColor']=WireColor
    #WireCSA
    WireCSA =groupped2.loc[i,'Wire WireCSA']
    connectors[connector][pin]['WireCSA']=WireCSA
    #WireCSA
    Multicore =groupped2.loc[i,'MulticoreInnerToOutter1']
    connectors[connector][pin]['Multicore']=Multicore
    
    #signal_category Group
    signal_group =groupped2.loc[i,'signal_cat']
    connectors[connector][pin]['signal_group']=signal_group
    # internal connector / external  connertor  internal_pin
    internal_pin = groupped2.loc[i, 'internal_pin']
    connectors[connector][pin]['internal_pin'] = internal_pin
    #signal_category Group
    try:
        distance_to_center =groupped2.loc[i,'distance_to_center']
        neighbors =groupped2.loc[i,'neighbors']
        connectors[connector][pin]['distance_to_center']=distance_to_center
        connectors[connector][pin]['neighbors']=neighbors
    except:
        connectors[connector][pin]['distance_to_center']=0
        connectors[connector][pin]['neighbors']=[]


In [40]:
eval(connectors['TO_184_F/171']['1']['dicc_signals']).keys()

dict_keys([0, 1, 2, 3, 4])

In [98]:
connectors['TO_184_F/171']['1']

{'PreferredSignal': 'na',
 'Cat_PreferredSignal': 6,
 'Signal': 'EMB_FUL_DISP_IN_MIR_030',
 'Cat_Signal': 0,
 'dicc_signals': "{0: 'EMB_FUL_DISP_IN_MIR_030', 1: 'EMB_FUL_DISP_IN_MIR_031', 2: 'EMB_FUL_DISP_IN_MIR_032', 3: 'EMB_FUL_DISP_IN_MIR_033', 4: 'EMB_FUL_DISP_IN_MIR_034'}",
 'check_signal': 0,
 'Num_Pins': 5,
 'Num_Pin_unique': 5,
 'more_signal_in_pin': False,
 'WireColor': 'W',
 'WireCSA': 1.0,
 'Multicore': 'MC142268',
 'signal_group': 7,
 'internal_pin': 'False',
 'distance_to_center': 11747.160380279141,
 'neighbors': ['3', '2', '4']}

In [97]:
len(connectors['TO_184_F/171'].keys())-1

5

In [ ]:
Cat_PreferredSignals=[]
for key in connectors['TO_111_LH_(121)'].keys():
    if key != 'PartNumber':
        Cat_PreferredSignals.append(connectors['TO_111_LH_(121)'][key]['Cat_PreferredSignal'])
        

In [ ]:
Cat_PreferredSignals

In [ ]:
import random

print(random.randint(0, max_categories ))

In [ ]:
for i in range(10):
    print(random.random())

In [ ]:
len(connectors.keys())

In [ ]:
len(data['Signal Name'].unique()),len(data['Pin PreferredSignal'].unique())

In [ ]:
signal_view=data.groupby(["Pin PreferredSignal"]).agg({"Pin Name":[ "count", 'nunique'], "Connector Name":'nunique' })
signal_view.head(10)

In [ ]:
groupped.to_csv("data/connectors.csv", index=False)

In [ ]:
groupped['Connector PartNumber'].fillna("NOPARTNUMBER", inplace=True)
groupped.head(20)